# Batch API Feasibility Check — HOSE

Notebook kiểm tra khả năng xây dựng batch pipeline cho dữ liệu HOSE:

- Thu thập OHLCV daily cho một nhóm mã thử nghiệm.
- Kiểm tra chất lượng dữ liệu Bronze.
- Chuẩn hóa dữ liệu Silver.
- Sinh trực tiếp `dim_date` ở Gold.
- Tạo `dim_symbol` cho 5 mã thử nghiệm.
- Tính chỉ báo và dựng thử `fact_hose_daily_market`.
- Xuất dữ liệu mẫu để chuẩn bị triển khai MinIO + Iceberg.

> Giả định file notebook nằm trong thư mục `notebooks/` của project.

## 1. Cài đặt thư viện

Chỉ chạy cell dưới nếu môi trường chưa có đủ package.

In [1]:
# %pip install -q vnstock pandas polars holidays pyarrow plotly

## 2. Import thư viện

In [2]:
from __future__ import annotations

from datetime import date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo
from typing import Iterable

import holidays
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from IPython.display import display

from vnstock.api.quote import Quote
from vnstock.api.listing import Listing
from vnstock.api.company import Company

## 3. Cấu hình project và phạm vi kiểm thử

Nếu notebook được mở từ thư mục `notebooks/`, dữ liệu đầu ra sẽ được lưu tại:

```text
project_root/
├── notebooks/
│   └── 01_api_feasibility_check_batch_clean.ipynb
└── data/
    └── feasibility/
```

In [3]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name.lower() in {"notebook", "notebooks"}
    else NOTEBOOK_DIR
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "feasibility"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SYMBOLS = ["FPT", "VCB", "HPG", "VNM", "MWG"]

OHLCV_START_DATE = "2026-02-01"
OHLCV_END_DATE = "2026-05-31"

DIM_DATE_START = date(2024, 1, 1)
DIM_DATE_END = date(2026, 12, 31)

SOURCE = "VCI"
EXCHANGE_CODE = "HOSE"
TIMEZONE = ZoneInfo("Asia/Ho_Chi_Minh")

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)

Notebook directory: c:\MiniProject\notebook
Project root: c:\MiniProject
Output directory: c:\MiniProject\data\feasibility


## 4. Hàm hỗ trợ

In [4]:
def first_existing_column(
    dataframe: pd.DataFrame,
    candidates: Iterable[str],
) -> str | None:
    for column in candidates:
        if column in dataframe.columns:
            return column
    return None


def normalize_symbol_list(result: object) -> pd.DataFrame:
    if isinstance(result, pd.DataFrame):
        dataframe = result.copy()
    elif isinstance(result, dict):
        dataframe = pd.DataFrame(result)
    elif isinstance(result, (list, tuple, set)):
        dataframe = pd.DataFrame({"symbol": list(result)})
    else:
        raise TypeError(f"Không hỗ trợ kiểu dữ liệu: {type(result)}")

    symbol_column = first_existing_column(
        dataframe,
        ["symbol", "ticker", "code"],
    )

    if symbol_column is None:
        raise ValueError(
            f"Không tìm thấy cột symbol. Các cột hiện có: {list(dataframe.columns)}"
        )

    if symbol_column != "symbol":
        dataframe = dataframe.rename(columns={symbol_column: "symbol"})

    dataframe["symbol"] = (
        dataframe["symbol"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return dataframe.drop_duplicates(subset=["symbol"]).reset_index(drop=True)

# Phần A — OHLCV Daily

## 5. Thu thập dữ liệu OHLCV thử nghiệm

Notebook chỉ lấy OHLCV cho `TEST_SYMBOLS` để kiểm tra API và pipeline.  
Sau khi MVP ổn định có thể mở rộng sang toàn bộ mã HOSE.

In [5]:
ohlcv_frames: list[pd.DataFrame] = []
api_logs: list[dict] = []

for symbol in TEST_SYMBOLS:
    try:
        quote = Quote(symbol=symbol, source=SOURCE)

        raw_df = quote.history(
            start=OHLCV_START_DATE,
            end=OHLCV_END_DATE,
            interval="1D",
        )

        if raw_df is None or raw_df.empty:
            raise ValueError("API không trả dữ liệu")

        df = raw_df.copy()
        df["time"] = pd.to_datetime(df["time"], errors="coerce")

        df = df[
            df["time"].between(
                pd.Timestamp(OHLCV_START_DATE),
                pd.Timestamp(OHLCV_END_DATE),
                inclusive="both",
            )
        ].copy()

        if df.empty:
            raise ValueError("Không còn dữ liệu sau khi lọc khoảng ngày")

        df["symbol"] = symbol
        df["exchange"] = EXCHANGE_CODE
        df["source"] = SOURCE
        df["ingested_at"] = pd.Timestamp.now(tz=TIMEZONE)

        ohlcv_frames.append(df)

        api_logs.append({
            "symbol": symbol,
            "status": "OK",
            "rows": len(df),
            "min_date": df["time"].min(),
            "max_date": df["time"].max(),
            "error": None,
        })

    except Exception as exc:
        api_logs.append({
            "symbol": symbol,
            "status": "ERROR",
            "rows": 0,
            "min_date": None,
            "max_date": None,
            "error": str(exc),
        })

api_log_df = pd.DataFrame(api_logs)
display(api_log_df)

if not ohlcv_frames:
    raise RuntimeError("Không thu thập được OHLCV cho bất kỳ mã nào.")

bronze_ohlcv_df = (
    pd.concat(ohlcv_frames, ignore_index=True)
    .sort_values(["symbol", "time"])
    .reset_index(drop=True)
)

display(bronze_ohlcv_df.head())

,symbol,status,rows,min_date,max_date,error
0,FPT,OK,77,2026-02-02,2026-05-29,None
1,VCB,OK,77,2026-02-02,2026-05-29,None
2,HPG,OK,77,2026-02-02,2026-05-29,None
3,VNM,OK,77,2026-02-02,2026-05-29,None
4,MWG,OK,77,2026-02-02,2026-05-29,None


,time,open,high,low,close,volume,symbol,exchange,source,ingested_at
0,2026-02-02,101.90,102.88,100.22,102.88,9362784,FPT,HOSE,VCI,2026-06-15 21:54:45.865328+07:00
1,2026-02-03,103.47,103.97,101.50,102.49,9116438,FPT,HOSE,VCI,2026-06-15 21:54:45.865328+07:00
2,2026-02-04,101.70,102.09,100.12,100.51,11665250,FPT,HOSE,VCI,2026-06-15 21:54:45.865328+07:00
3,2026-02-05,99.82,100.51,97.65,97.65,18118564,FPT,HOSE,VCI,2026-06-15 21:54:45.865328+07:00
4,2026-02-06,96.67,98.64,95.38,96.27,9796115,FPT,HOSE,VCI,2026-06-15 21:54:45.865328+07:00


## 6. Khảo sát độ phủ dữ liệu Bronze

In [6]:
coverage_df = (
    bronze_ohlcv_df.groupby("symbol", as_index=False)
    .agg(
        rows=("time", "size"),
        min_date=("time", "min"),
        max_date=("time", "max"),
        trading_days=("time", "nunique"),
    )
)

display(coverage_df)
print("Shape:", bronze_ohlcv_df.shape)
print("Columns:", list(bronze_ohlcv_df.columns))

,symbol,rows,min_date,max_date,trading_days
0,FPT,77,2026-02-02,2026-05-29,77
1,HPG,77,2026-02-02,2026-05-29,77
2,MWG,77,2026-02-02,2026-05-29,77
3,VCB,77,2026-02-02,2026-05-29,77
4,VNM,77,2026-02-02,2026-05-29,77


Shape: (385, 10)
Columns: ['time', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'exchange', 'source', 'ingested_at']


## 7. Kiểm tra chất lượng Bronze OHLCV

In [27]:
required_columns = [
    "time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "symbol",
    "exchange",
    "source",
    "ingested_at",
]

missing_columns = sorted(
    set(required_columns) - set(bronze_ohlcv_df.columns)
)

quality_checks = [
    {
        "check_name": "required columns",
        "error_count": len(missing_columns),
        "detail": ", ".join(missing_columns) if missing_columns else "PASS",
    },
    {
        "check_name": "null required fields",
        "error_count": (
            int(bronze_ohlcv_df[required_columns].isna().any(axis=1).sum())
            if not missing_columns
            else len(bronze_ohlcv_df)
        ),
        "detail": None,
    },
    {
        "check_name": "duplicate symbol + time",
        "error_count": int(
            bronze_ohlcv_df.duplicated(
                subset=["symbol", "time"]
            ).sum()
        ),
        "detail": None,
    },
    {
        "check_name": "non-positive OHLC",
        "error_count": int(
            (
                (bronze_ohlcv_df["open"] <= 0)
                | (bronze_ohlcv_df["high"] <= 0)
                | (bronze_ohlcv_df["low"] <= 0)
                | (bronze_ohlcv_df["close"] <= 0)
            ).sum()
        ),
        "detail": None,
    },
    {
        "check_name": "negative volume",
        "error_count": int((bronze_ohlcv_df["volume"] < 0).sum()),
        "detail": None,
    },
    {
        "check_name": "invalid OHLC range",
        "error_count": int(
            (
                (bronze_ohlcv_df["high"] < bronze_ohlcv_df["low"])
                | (bronze_ohlcv_df["high"] < bronze_ohlcv_df["open"])
                | (bronze_ohlcv_df["high"] < bronze_ohlcv_df["close"])
                | (bronze_ohlcv_df["low"] > bronze_ohlcv_df["open"])
                | (bronze_ohlcv_df["low"] > bronze_ohlcv_df["close"])
            ).sum()
        ),
        "detail": None,
    },
]

ohlcv_quality_df = pd.DataFrame(quality_checks)
display(ohlcv_quality_df)

total_ohlcv_errors = int(ohlcv_quality_df["error_count"].sum())
print("Tổng số lỗi:", total_ohlcv_errors)

,check_name,error_count,detail
0,required columns,0,PASS
1,null required fields,0,None
2,duplicate symbol + time,0,None
3,non-positive OHLC,0,None
4,negative volume,0,None
5,invalid OHLC range,0,None


Tổng số lỗi: 0


## 8. Tạo Silver OHLCV

In [28]:
silver_ohlcv_df = (
    bronze_ohlcv_df.rename(columns={
        "time": "trading_date",
        "open": "open_price",
        "high": "high_price",
        "low": "low_price",
        "close": "close_price",
        "exchange": "exchange_code",
    })
    .copy()
)

silver_ohlcv_df["trading_date"] = pd.to_datetime(
    silver_ohlcv_df["trading_date"],
    errors="raise",
)

numeric_columns = [
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
]

for column in numeric_columns:
    silver_ohlcv_df[column] = pd.to_numeric(
        silver_ohlcv_df[column],
        errors="raise",
    )

silver_ohlcv_df["volume"] = silver_ohlcv_df["volume"].astype("int64")

silver_ohlcv_df = (
    silver_ohlcv_df[
        [
            "trading_date",
            "symbol",
            "exchange_code",
            "open_price",
            "high_price",
            "low_price",
            "close_price",
            "volume",
            "source",
            "ingested_at",
        ]
    ]
    .drop_duplicates(subset=["symbol", "trading_date"])
    .sort_values(["symbol", "trading_date"])
    .reset_index(drop=True)
)

display(silver_ohlcv_df.head())

,trading_date,symbol,exchange_code,open_price,high_price,low_price,close_price,volume,source,ingested_at
0,2026-02-02,FPT,HOSE,101.90,102.88,100.22,102.88,9362784,VCI,2026-06-13 22:23:54.347245+07:00
1,2026-02-03,FPT,HOSE,103.47,103.97,101.50,102.49,9116438,VCI,2026-06-13 22:23:54.347245+07:00
2,2026-02-04,FPT,HOSE,101.70,102.09,100.12,100.51,11665250,VCI,2026-06-13 22:23:54.347245+07:00
3,2026-02-05,FPT,HOSE,99.82,100.51,97.65,97.65,18118564,VCI,2026-06-13 22:23:54.347245+07:00
4,2026-02-06,FPT,HOSE,96.67,98.64,95.38,96.27,9796115,VCI,2026-06-13 22:23:54.347245+07:00


# Phần B — Gold `dim_date`

## 9. Sinh calendar 2024–2026

`dim_date` được tạo trực tiếp ở Gold, không tạo `bronze_calendar` hoặc `silver_calendar`.

In [7]:
dim_date_base = (
    pl.DataFrame({
        "full_date": pl.date_range(
            start=DIM_DATE_START,
            end=DIM_DATE_END,
            interval="1d",
            eager=True,
        )
    })
    .with_columns([
        pl.col("full_date").dt.day().cast(pl.UInt8).alias("day"),
        pl.col("full_date").dt.weekday().cast(pl.UInt8).alias("day_of_week"),
        pl.col("full_date").dt.week().cast(pl.UInt8).alias("cal_week"),
        pl.col("full_date").dt.month().cast(pl.UInt8).alias("cal_month"),
        pl.col("full_date").dt.quarter().cast(pl.UInt8).alias("cal_quarter"),
        pl.col("full_date").dt.year().cast(pl.UInt16).alias("cal_year"),
    ])
    .with_columns(
        pl.col("day_of_week")
        .is_in([6, 7])
        .cast(pl.UInt8)
        .alias("is_weekend")
    )
)

display(dim_date_base.head())

full_date,day,day_of_week,cal_week,cal_month,cal_quarter,cal_year,is_weekend
date,u8,u8,u8,u8,u8,u16,u8
2024-01-01,1,1,1,1,1,2024,0
2024-01-02,2,2,1,1,1,2024,0
2024-01-03,3,3,1,1,1,2024,0
2024-01-04,4,4,1,1,1,2024,0
2024-01-05,5,5,1,1,1,2024,0


## 10. Sinh ngày lễ Việt Nam

In [8]:
vn_holidays = holidays.country_holidays(
    country="VN",
    years=range(DIM_DATE_START.year, DIM_DATE_END.year + 1),
    language="vi",
    observed=True,
)

holiday_ingested_at = datetime.now(TIMEZONE)

holiday_rows = [
    {
        "full_date": holiday_date,
        "event_name": str(holiday_name),
        "event_type": (
            "Compensation"
            if (
                "nghỉ bù" in str(holiday_name).lower()
                or "hoán đổi" in str(holiday_name).lower()
            )
            else "Holiday"
        ),
        "event_ingested_at": holiday_ingested_at,
    }
    for holiday_date, holiday_name in sorted(vn_holidays.items())
]

holiday_df = pl.DataFrame(
    holiday_rows,
    schema={
        "full_date": pl.Date,
        "event_name": pl.String,
        "event_type": pl.String,
        "event_ingested_at": pl.Datetime(
            time_zone="Asia/Ho_Chi_Minh"
        ),
    },
)

display(
    holiday_df.filter(
        pl.col("full_date").dt.year() == 2026
    ).sort("full_date")
)

full_date,event_name,event_type,event_ingested_at
date,str,str,"datetime[μs, Asia/Ho_Chi_Minh]"
2026-01-01,"""Tết Dương lịch""","""Holiday""",2026-06-15 21:55:15.283104 +07
2026-02-16,"""Giao thừa Tết Nguyên Đán""","""Holiday""",2026-06-15 21:55:15.283104 +07
2026-02-17,"""Tết Nguyên Đán""","""Holiday""",2026-06-15 21:55:15.283104 +07
2026-02-18,"""Mùng hai Tết Nguyên Đán""","""Holiday""",2026-06-15 21:55:15.283104 +07
2026-02-19,"""Mùng ba Tết Nguyên Đán""","""Holiday""",2026-06-15 21:55:15.283104 +07
…,…,…,…
2026-04-27,"""Ngày Giỗ Tổ Hùng Vương (nghỉ b…","""Compensation""",2026-06-15 21:55:15.283104 +07
2026-04-30,"""Ngày Chiến thắng""","""Holiday""",2026-06-15 21:55:15.283104 +07
2026-05-01,"""Ngày Quốc tế Lao động""","""Holiday""",2026-06-15 21:55:15.283104 +07


## 11. Merge calendar với ngày lễ và tạo khóa

In [31]:
holiday_daily = (
    holiday_df.group_by("full_date")
    .agg([
        pl.col("event_name")
        .unique()
        .sort()
        .str.join(" | ")
        .alias("event_name"),

        pl.col("event_type")
        .unique()
        .sort()
        .str.join(" | ")
        .alias("event_type"),

        pl.col("event_ingested_at")
        .max()
        .alias("event_ingested_at"),
    ])
)

dim_date = (
    dim_date_base
    .join(
        holiday_daily,
        on="full_date",
        how="left",
    )
    .with_columns(
        (
            (pl.col("is_weekend") == 1)
            | pl.col("event_name").is_not_null()
        )
        .cast(pl.UInt8)
        .alias("is_day_off")
    )
    .sort("full_date")
    .with_row_index(
        name="date_key",
        offset=1,
    )
    .with_columns([
        pl.col("date_key").cast(pl.UInt32),

        pl.col("full_date")
        .dt.strftime("%Y%m%d")
        .cast(pl.UInt32)
        .alias("date_id"),

        pl.lit(datetime.now(TIMEZONE))
        .cast(pl.Datetime(time_zone="Asia/Ho_Chi_Minh"))
        .alias("updated_at"),
    ])
    .select([
        "date_key",
        "date_id",
        "full_date",
        "day",
        "day_of_week",
        "cal_week",
        "cal_month",
        "cal_quarter",
        "cal_year",
        "is_weekend",
        "event_name",
        "event_type",
        "is_day_off",
        "event_ingested_at",
        "updated_at",
    ])
)

display(dim_date.head())

date_key,date_id,full_date,day,day_of_week,cal_week,cal_month,cal_quarter,cal_year,is_weekend,event_name,event_type,is_day_off,event_ingested_at,updated_at
u32,u32,date,u8,u8,u8,u8,u8,u16,u8,str,str,u8,"datetime[μs, Asia/Ho_Chi_Minh]","datetime[μs, Asia/Ho_Chi_Minh]"
1,20240101,2024-01-01,1,1,1,1,1,2024,0,"""Tết Dương lịch""","""Holiday""",1,2026-06-13 22:24:12.174144 +07,2026-06-13 22:24:35.456738 +07
2,20240102,2024-01-02,2,2,1,1,1,2024,0,null,null,0,null,2026-06-13 22:24:35.456738 +07
3,20240103,2024-01-03,3,3,1,1,1,2024,0,null,null,0,null,2026-06-13 22:24:35.456738 +07
4,20240104,2024-01-04,4,4,1,1,1,2024,0,null,null,0,null,2026-06-13 22:24:35.456738 +07
5,20240105,2024-01-05,5,5,1,1,1,2024,0,null,null,0,null,2026-06-13 22:24:35.456738 +07


## 12. Kiểm tra chất lượng `dim_date`

In [32]:
expected_days = (DIM_DATE_END - DIM_DATE_START).days + 1

assert dim_date.height == expected_days
assert dim_date["date_key"].n_unique() == dim_date.height
assert dim_date["date_id"].n_unique() == dim_date.height
assert dim_date["full_date"].n_unique() == dim_date.height

assert dim_date["date_key"].null_count() == 0
assert dim_date["date_id"].null_count() == 0
assert dim_date["full_date"].null_count() == 0
assert dim_date["is_day_off"].null_count() == 0

assert dim_date["full_date"].min() == DIM_DATE_START
assert dim_date["full_date"].max() == DIM_DATE_END

print("dim_date validation passed")
print("Số dòng:", dim_date.height)
print(
    "Số ngày có sự kiện:",
    dim_date.filter(
        pl.col("event_name").is_not_null()
    ).height,
)

dim_date validation passed
Số dòng: 1096
Số ngày có sự kiện: 38


# Phần C — Chỉ báo kỹ thuật và Gold Fact

## 13. Tính chỉ báo kỹ thuật

Đây là bước thử nghiệm full-history. Khi đưa vào pipeline thật, fact sẽ được cập nhật incremental theo ngày.

In [33]:
indicator_df = (
    silver_ohlcv_df
    .sort_values(["symbol", "trading_date"])
    .reset_index(drop=True)
    .copy()
)

grouped = indicator_df.groupby("symbol", group_keys=False)

indicator_df["price_change"] = grouped["close_price"].diff()
indicator_df["pct_change"] = grouped["close_price"].pct_change() * 100

indicator_df["sma20"] = grouped["close_price"].transform(
    lambda series: series.rolling(
        window=20,
        min_periods=20,
    ).mean()
)

indicator_df["ema20"] = grouped["close_price"].transform(
    lambda series: series.ewm(
        span=20,
        adjust=False,
        min_periods=20,
    ).mean()
)

indicator_df["avg_volume_20d"] = grouped["volume"].transform(
    lambda series: series.rolling(
        window=20,
        min_periods=20,
    ).mean()
)


def calculate_rsi(
    series: pd.Series,
    period: int = 14,
) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period,
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period,
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


indicator_df["rsi14"] = grouped["close_price"].transform(
    calculate_rsi
)

ema12 = grouped["close_price"].transform(
    lambda series: series.ewm(
        span=12,
        adjust=False,
        min_periods=12,
    ).mean()
)

ema26 = grouped["close_price"].transform(
    lambda series: series.ewm(
        span=26,
        adjust=False,
        min_periods=26,
    ).mean()
)

indicator_df["macd"] = ema12 - ema26

display(indicator_df.head(25))

,trading_date,symbol,exchange_code,open_price,high_price,low_price,close_price,volume,source,ingested_at,price_change,pct_change,sma20,ema20,avg_volume_20d,rsi14,macd
0,2026-02-02,FPT,HOSE,101.90,102.88,100.22,102.88,9362784,VCI,2026-06-13 22:23:54.347245+07:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-02-03,FPT,HOSE,103.47,103.97,101.50,102.49,9116438,VCI,2026-06-13 22:23:54.347245+07:00,-0.39,-0.379082,NaN,NaN,NaN,NaN,NaN
2,2026-02-04,FPT,HOSE,101.70,102.09,100.12,100.51,11665250,VCI,2026-06-13 22:23:54.347245+07:00,-1.98,-1.931896,NaN,NaN,NaN,NaN,NaN
3,2026-02-05,FPT,HOSE,99.82,100.51,97.65,97.65,18118564,VCI,2026-06-13 22:23:54.347245+07:00,-2.86,-2.845488,NaN,NaN,NaN,NaN,NaN
4,2026-02-06,FPT,HOSE,96.67,98.64,95.38,96.27,9796115,VCI,2026-06-13 22:23:54.347245+07:00,-1.38,-1.413210,NaN,NaN,NaN,NaN,NaN
5,2026-02-09,FPT,HOSE,97.46,98.05,96.57,97.65,5123453,VCI,2026-06-13 22:23:54.347245+07:00,1.38,1.433468,NaN,NaN,NaN,NaN,NaN
6,2026-02-10,FPT,HOSE,98.54,98.54,95.98,96.47,6143120,VCI,2026-06-13 22:23:54.347245+07:00,-1.18,-1.208397,NaN,NaN,NaN,NaN,NaN
7,2026-02-11,FPT,HOSE,96.86,98.15,96.08,97.26,5525584,VCI,2026-06-13 22:23:54.347245+07:00,0.79,0.818907,NaN,NaN,NaN,NaN,NaN
8,2026-02-12,FPT,HOSE,97.65,97.85,97.06,97.46,4137993,VCI,2026-06-13 22:23:54.347245+07:00,0.20,0.205634,NaN,NaN,NaN,NaN,NaN
9,2026-02-13,FPT,HOSE,96.77,97.16,94.99,94.99,18546292,VCI,2026-06-13 22:23:54.347245+07:00,-2.47,-2.534373,NaN,NaN,NaN,NaN,NaN


## 14. Tạo thử `fact_hose_daily_market`

In [34]:
date_lookup_df = dim_date.select(
    ["date_key", "full_date"]
).to_pandas()

date_lookup_df["full_date"] = pd.to_datetime(
    date_lookup_df["full_date"]
)

fact_hose_daily_market = (
    indicator_df.merge(
        date_lookup_df,
        left_on="trading_date",
        right_on="full_date",
        how="left",
        validate="many_to_one",
    )
    .drop(columns=["full_date"])
)

fact_hose_daily_market["updated_at"] = pd.Timestamp.now(tz=TIMEZONE)

display(fact_hose_daily_market.head())

,trading_date,symbol,exchange_code,open_price,high_price,low_price,close_price,volume,source,ingested_at,price_change,pct_change,sma20,ema20,avg_volume_20d,rsi14,macd,date_key,updated_at
0,2026-02-02,FPT,HOSE,101.90,102.88,100.22,102.88,9362784,VCI,2026-06-13 22:23:54.347245+07:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,764,2026-06-13 22:25:08.836246+07:00
1,2026-02-03,FPT,HOSE,103.47,103.97,101.50,102.49,9116438,VCI,2026-06-13 22:23:54.347245+07:00,-0.39,-0.379082,NaN,NaN,NaN,NaN,NaN,765,2026-06-13 22:25:08.836246+07:00
2,2026-02-04,FPT,HOSE,101.70,102.09,100.12,100.51,11665250,VCI,2026-06-13 22:23:54.347245+07:00,-1.98,-1.931896,NaN,NaN,NaN,NaN,NaN,766,2026-06-13 22:25:08.836246+07:00
3,2026-02-05,FPT,HOSE,99.82,100.51,97.65,97.65,18118564,VCI,2026-06-13 22:23:54.347245+07:00,-2.86,-2.845488,NaN,NaN,NaN,NaN,NaN,767,2026-06-13 22:25:08.836246+07:00
4,2026-02-06,FPT,HOSE,96.67,98.64,95.38,96.27,9796115,VCI,2026-06-13 22:23:54.347245+07:00,-1.38,-1.413210,NaN,NaN,NaN,NaN,NaN,768,2026-06-13 22:25:08.836246+07:00


# Phần D — Gold `dim_symbol`

## 15. Lấy danh sách mã HOSE

`dim_symbol` trong notebook feasibility chỉ được tạo cho 5 mã thử nghiệm.  
Cell dưới thử các method phổ biến của `Listing`; nếu phiên bản `vnstock` khác API, notebook sẽ hiển thị method khả dụng để chỉnh lại.

In [35]:
listing = Listing(source=SOURCE)

hose_listing_raw = None
used_listing_method = None

listing_method_candidates = [
    ("symbols_by_exchange", (EXCHANGE_CODE,)),
    ("symbols_by_group", (EXCHANGE_CODE,)),
    ("all_symbols", tuple()),
]

for method_name, args in listing_method_candidates:
    if not hasattr(listing, method_name):
        continue

    try:
        candidate = getattr(listing, method_name)(*args)
        candidate_df = normalize_symbol_list(candidate)

        if candidate_df.empty:
            continue

        hose_listing_raw = candidate_df
        used_listing_method = method_name
        break

    except Exception:
        continue

if hose_listing_raw is None:
    available_methods = [
        method_name
        for method_name in dir(listing)
        if not method_name.startswith("_")
    ]

    raise RuntimeError(
        "Không tìm thấy method phù hợp để lấy danh sách HOSE. "
        f"Các method hiện có: {available_methods}"
    )

hose_listing_raw = (
    hose_listing_raw[
        hose_listing_raw["symbol"].isin(TEST_SYMBOLS)
    ]
    .drop_duplicates(subset=["symbol"])
    .sort_values("symbol")
    .reset_index(drop=True)
)

missing_test_symbols = sorted(
    set(TEST_SYMBOLS) - set(hose_listing_raw["symbol"])
)

if missing_test_symbols:
    raise ValueError(
        f"Không tìm thấy các mã trong Listing: {missing_test_symbols}"
    )

print("Listing method:", used_listing_method)
print("Số symbol dùng trong dim_symbol:", len(hose_listing_raw))
display(hose_listing_raw)

Listing method: all_symbols
Số symbol dùng trong dim_symbol: 5


,symbol,organ_name
0,FPT,Công ty Cổ phần FPT
1,HPG,Công ty Cổ phần Tập đoàn Hòa Phát
2,MWG,Công ty Cổ phần Đầu tư Thế Giới Di Động
3,VCB,Ngân hàng Thương mại Cổ phần Ngoại thương Việt...
4,VNM,Công ty Cổ phần Sữa Việt Nam


## 16. Tạo bản `dim_symbol` bootstrap

Để notebook feasibility chạy nhanh, phần gọi `Company` chi tiết chỉ thử cho `TEST_SYMBOLS`.  
Chỉ 5 mã trong `TEST_SYMBOLS` được đưa vào dimension để tránh tạo nhiều dòng thiếu metadata.

In [36]:
# hose_listing_raw đã được đảm bảo non-None ở cell trước (raise nếu None)
assert hose_listing_raw is not None

listing_company_column = first_existing_column(
    hose_listing_raw,
    ["organ_name", "company_name", "organ_short_name"],
)

dim_symbol_base = hose_listing_raw.copy()

if listing_company_column is not None:
    dim_symbol_base = dim_symbol_base.rename(
        columns={listing_company_column: "company_name"}
    )
else:
    dim_symbol_base["company_name"] = None

company_rows: list[pd.DataFrame] = []
company_logs: list[dict] = []

company_method_candidates = [
    "overview",
    "profile",
    "company_profile",
    "info",
]

for symbol in TEST_SYMBOLS:
    try:
        company = Company(symbol=symbol, source=SOURCE)
        result_df = None
        used_method = None

        for method_name in company_method_candidates:
            if not hasattr(company, method_name):
                continue

            try:
                result = getattr(company, method_name)()

                if isinstance(result, pd.DataFrame):
                    candidate_df = result.copy()
                elif isinstance(result, dict):
                    candidate_df = pd.DataFrame([result])
                else:
                    continue

                if not candidate_df.empty:
                    result_df = candidate_df
                    used_method = method_name
                    break

            except Exception:
                continue

        if result_df is None:
            raise ValueError("Không có Company method nào trả dữ liệu")

        result_df["symbol"] = symbol
        company_rows.append(result_df)

        company_logs.append({
            "symbol": symbol,
            "status": "OK",
            "method": used_method,
            "error": None,
        })

    except Exception as exc:
        company_logs.append({
            "symbol": symbol,
            "status": "ERROR",
            "method": None,
            "error": str(exc),
        })

company_log_df = pd.DataFrame(company_logs)

company_raw_df = (
    pd.concat(company_rows, ignore_index=True)
    if company_rows
    else pd.DataFrame()
)

display(company_log_df)

,symbol,status,method,error
0,FPT,OK,overview,None
1,VCB,OK,overview,None
2,HPG,OK,overview,None
3,VNM,OK,overview,None
4,MWG,OK,overview,None


## 17. Chuẩn hóa `dim_symbol`

In [38]:
metadata_small = pd.DataFrame({"symbol": TEST_SYMBOLS})

if not company_raw_df.empty:
    metadata_mapping = {
        "company_name_meta": first_existing_column(
            company_raw_df,
            ["organ_name", "organ_short_name"],
        ),
        "sector_name": first_existing_column(
            company_raw_df,
            ["sector"],
        ),
        "company_profile": first_existing_column(
            company_raw_df,
            ["company_profile", "profile"],
        ),
        "listing_date": first_existing_column(
            company_raw_df,
            ["listing_date"],
        ),
        "issued_share": first_existing_column(
            company_raw_df,
            ["issued_share"],
        ),
    }

    metadata_small = company_raw_df[["symbol"]].copy()

    for output_column, source_column in metadata_mapping.items():
        metadata_small[output_column] = (
            company_raw_df[source_column]
            if source_column is not None
            else None
        )

    metadata_small = metadata_small.drop_duplicates(
        subset=["symbol"]
    )

dim_symbol = dim_symbol_base.merge(
    metadata_small,
    on="symbol",
    how="left",
)

if "company_name_meta" in dim_symbol.columns:
    dim_symbol["company_name"] = dim_symbol["company_name"].fillna(
        dim_symbol["company_name_meta"]
    )

for column in [
    "sector_name",
    "company_profile",
    "listing_date",
    "issued_share",
]:
    if column not in dim_symbol.columns:
        dim_symbol[column] = None

dim_symbol = (
    dim_symbol
    .drop_duplicates(subset=["symbol"])
    .sort_values("symbol")
    .reset_index(drop=True)
)

dim_symbol.insert(
    0,
    "symbol_key",
    np.arange(1, len(dim_symbol) + 1, dtype=np.uint32),
)

dim_symbol["exchange_code"] = EXCHANGE_CODE
dim_symbol["listed_status"] = "LISTED"
dim_symbol["updated_at"] = pd.Timestamp.now(tz=TIMEZONE)

dim_symbol = dim_symbol[
    [
        "symbol_key",
        "symbol",
        "company_name",
        "sector_name",
        "company_profile",
        "listing_date",
        "exchange_code",
        "listed_status",
        "updated_at",
    ]
]

display(dim_symbol.head())
print("Số symbol trong dim_symbol:", len(dim_symbol))
assert set(dim_symbol["symbol"]) == set(TEST_SYMBOLS)

,symbol_key,symbol,company_name,sector_name,company_profile,listing_date,exchange_code,listed_status,updated_at
0,1,FPT,Công ty Cổ phần FPT,Technology,Công ty Cổ phần FPT (FPT) có tiền thân là Công...,2006-12-13T00:00:00,HOSE,LISTED,2026-06-13 22:28:09.246571+07:00
1,2,HPG,Công ty Cổ phần Tập đoàn Hòa Phát,Basic Resources,Công ty Cổ phần Tập đoàn Hoà Phát (HPG) là một...,2007-11-15T00:00:00,HOSE,LISTED,2026-06-13 22:28:09.246571+07:00
2,3,MWG,Công ty Cổ phần Đầu tư Thế Giới Di Động,Retail,Công ty Cổ phần Đầu tư Thế Giới Di Động (MWG) ...,2014-07-14T00:00:00,HOSE,LISTED,2026-06-13 22:28:09.246571+07:00
3,4,VCB,Ngân hàng Thương mại Cổ phần Ngoại thương Việt...,Banks,Ngân hàng Thương mại Cổ phần Ngoại thương Việt...,2009-06-30T00:00:00,HOSE,LISTED,2026-06-13 22:28:09.246571+07:00
4,5,VNM,Công ty Cổ phần Sữa Việt Nam,Food & Beverage,Công ty Cổ phần Sữa Việt Nam (VNM) có tiền thâ...,2006-01-19T00:00:00,HOSE,LISTED,2026-06-13 22:28:09.246571+07:00


Số symbol trong dim_symbol: 5


## 18. Hoàn thiện fact bằng `symbol_key`

In [39]:
symbol_lookup_df = dim_symbol[
    ["symbol_key", "symbol"]
].copy()

fact_hose_daily_market = (
    fact_hose_daily_market.merge(
        symbol_lookup_df,
        on="symbol",
        how="left",
        validate="many_to_one",
    )
)

fact_hose_daily_market = (
    fact_hose_daily_market[
        [
            "symbol_key",
            "date_key",
            "trading_date",
            "open_price",
            "high_price",
            "low_price",
            "close_price",
            "volume",
            "price_change",
            "pct_change",
            "sma20",
            "ema20",
            "rsi14",
            "macd",
            "avg_volume_20d",
            "updated_at",
        ]
    ]
    .sort_values(["symbol_key", "trading_date"])
    .reset_index(drop=True)
)

assert fact_hose_daily_market["symbol_key"].notna().all()
assert fact_hose_daily_market["date_key"].notna().all()
assert not fact_hose_daily_market.duplicated(
    subset=["symbol_key", "date_key"]
).any()

display(fact_hose_daily_market.head())

,symbol_key,date_key,trading_date,open_price,high_price,low_price,close_price,volume,price_change,pct_change,sma20,ema20,rsi14,macd,avg_volume_20d,updated_at
0,1,764,2026-02-02,101.90,102.88,100.22,102.88,9362784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-13 22:25:08.836246+07:00
1,1,765,2026-02-03,103.47,103.97,101.50,102.49,9116438,-0.39,-0.379082,NaN,NaN,NaN,NaN,NaN,2026-06-13 22:25:08.836246+07:00
2,1,766,2026-02-04,101.70,102.09,100.12,100.51,11665250,-1.98,-1.931896,NaN,NaN,NaN,NaN,NaN,2026-06-13 22:25:08.836246+07:00
3,1,767,2026-02-05,99.82,100.51,97.65,97.65,18118564,-2.86,-2.845488,NaN,NaN,NaN,NaN,NaN,2026-06-13 22:25:08.836246+07:00
4,1,768,2026-02-06,96.67,98.64,95.38,96.27,9796115,-1.38,-1.413210,NaN,NaN,NaN,NaN,NaN,2026-06-13 22:25:08.836246+07:00


# Phần E — Kết quả và xuất dữ liệu

## 19. Tổng kết khả năng tạo bảng

In [40]:
summary_df = pd.DataFrame([
    {
        "target_table": "bronze_hose_ohlcv_daily",
        "status": "OK" if not bronze_ohlcv_df.empty else "MISSING",
        "note": "Raw OHLCV cho các mã kiểm thử",
    },
    {
        "target_table": "silver_hose_ohlcv_daily",
        "status": "OK" if total_ohlcv_errors == 0 else "CHECK",
        "note": "Đã chuẩn hóa schema và loại duplicate",
    },
    {
        "target_table": "dim_date",
        "status": "OK" if dim_date.height > 0 else "MISSING",
        "note": "Sinh trực tiếp ở Gold cho giai đoạn 2024-2026",
    },
    {
        "target_table": "dim_symbol",
        "status": "OK" if not dim_symbol.empty else "MISSING",
        "note": "Chỉ tạo cho 5 mã kiểm thử",
    },
    {
        "target_table": "fact_hose_daily_market",
        "status": (
            "OK"
            if not fact_hose_daily_market.empty
            else "MISSING"
        ),
        "note": "OHLCV + technical indicators + surrogate keys",
    },
])

display(summary_df)

,target_table,status,note
0,bronze_hose_ohlcv_daily,OK,Raw OHLCV cho các mã kiểm thử
1,silver_hose_ohlcv_daily,OK,Đã chuẩn hóa schema và loại duplicate
2,dim_date,OK,Sinh trực tiếp ở Gold cho giai đoạn 2024-2026
3,dim_symbol,OK,Chỉ tạo cho 5 mã kiểm thử
4,fact_hose_daily_market,OK,OHLCV + technical indicators + surrogate keys


## 20. Xuất dữ liệu mẫu

In [44]:
# Các DataFrame dùng Pandas
exports: dict[str, pd.DataFrame] = {
    "bronze_hose_ohlcv_daily_test": bronze_ohlcv_df,
    "silver_hose_ohlcv_daily_test": silver_ohlcv_df,
    "dim_symbol_test": dim_symbol,
    "fact_hose_daily_market_test": fact_hose_daily_market,
    "ohlcv_quality_check": ohlcv_quality_df,
    "api_feasibility_summary": summary_df,
}

for base_name, dataframe in exports.items():
    parquet_path = OUTPUT_DIR / f"{base_name}.parquet"
    csv_path = OUTPUT_DIR / f"{base_name}.csv"

    # Parquet cho pipeline
    dataframe.to_parquet(
        parquet_path,
        index=False,
    )

    # CSV để xem nhanh
    dataframe.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

# dim_date là Polars DataFrame
dim_date.write_parquet(
    OUTPUT_DIR / "dim_date_2024_2026.parquet",
    compression="zstd",
)

dim_date.write_csv(
    OUTPUT_DIR / "dim_date_2024_2026.csv",
)

print("Đã lưu dữ liệu tại:", OUTPUT_DIR)

for output_path in sorted(OUTPUT_DIR.iterdir()):
    print("-", output_path.name)

Đã lưu dữ liệu tại: c:\Users\Dang Quoc Cuong\OneDrive - 43htgs\Máy tính\UIT\DE\VDT\MiniProject\data\feasibility
- api_feasibility_summary.csv
- api_feasibility_summary.parquet
- bronze_hose_ohlcv_daily_test.csv
- bronze_hose_ohlcv_daily_test.parquet
- dim_date_2024_2026.csv
- dim_date_2024_2026.parquet
- dim_symbol_test.csv
- dim_symbol_test.parquet
- fact_hose_daily_market_test.csv
- fact_hose_daily_market_test.parquet
- ohlcv_quality_check.csv
- ohlcv_quality_check.parquet
- silver_hose_ohlcv_daily_test.csv
- silver_hose_ohlcv_daily_test.parquet


## 21. Kiểm tra biểu đồ nến

In [42]:
CHART_SYMBOL = "FPT"

chart_df = indicator_df[
    indicator_df["symbol"] == CHART_SYMBOL
].copy()

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=chart_df["trading_date"],
    open=chart_df["open_price"],
    high=chart_df["high_price"],
    low=chart_df["low_price"],
    close=chart_df["close_price"],
    name=CHART_SYMBOL,
))

fig.add_trace(go.Scatter(
    x=chart_df["trading_date"],
    y=chart_df["sma20"],
    mode="lines",
    name="SMA20",
))

fig.add_trace(go.Scatter(
    x=chart_df["trading_date"],
    y=chart_df["ema20"],
    mode="lines",
    name="EMA20",
))

fig.update_layout(
    title=f"Candlestick Chart — {CHART_SYMBOL}",
    xaxis_title="Trading Date",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
)

fig.show()

## 22. Kiểm tra khoảng trống giữa các phiên

In [43]:
gap_df = chart_df[["trading_date"]].copy()
gap_df["previous_trading_date"] = gap_df[
    "trading_date"
].shift(1)

gap_df["gap_days"] = (
    gap_df["trading_date"]
    - gap_df["previous_trading_date"]
).dt.days

display(
    gap_df[
        gap_df["gap_days"] > 3
    ].sort_values(
        "gap_days",
        ascending=False,
    )
)

,trading_date,previous_trading_date,gap_days
10,2026-02-23,2026-02-13,10.0
57,2026-05-04,2026-04-29,5.0
55,2026-04-28,2026-04-24,4.0


## Kết luận

Notebook đã kiểm tra được luồng dữ liệu:

```text
VNStock API
→ Bronze OHLCV
→ Silver OHLCV
→ Gold dim_date / dim_symbol / fact_hose_daily_market
→ ClickHouse
→ Streamlit
```

Trong pipeline thực tế:

- `dim_date` được tạo sẵn và chỉ mở rộng khi cần thêm năm.
- `dim_symbol` được bootstrap từ toàn bộ HOSE và refresh định kỳ.
- `fact_hose_daily_market` được cập nhật incremental theo ngày.
- Iceberg sẽ dùng partition ngày và dynamic partition overwrite để bảo đảm tính lũy đẳng.